# JEOL elemental maps (.pts)

Import required commands and define path were demo data is stored. Adapt this for your installation.

In [ ]:
from pathlib import PurePath
import numpy as np
import matplotlib.pyplot as plt
from JEOL_eds import JEOL_pts
import JEOL_eds.utils as JU

base = PurePath("/home/alxneit/private/src/JEOL_eds")

Load example data (`128.pts`). This data set contains 50 frames with size 128x128 and 4000 energy channels. With the default data type (`uint16`) this results in about 6.5 GB! On top of this, there are also 50 images stored (one per frame) that can be used e.g. to compensate for sample movement. To minimize memory consumption, we first load only the meta data that contain the reference spectrum (sum over all frames and positions) and plot it to identify the maximum energy that contains relevant data.

In [ ]:
dc = JEOL_pts(base / "data" / "128.pts", only_metadata=True)
JU.plot_spectrum(dc.ref_spectrum, log_y=True)

We use an energy cutoff at 12keV. This reduces the memory requirement by about 2/3. To further reduce the memory consumption we could rebin (`rebin=(2,2)` the data while loading.

Load the data (individual frames) including the drift images. The data cube will contain 50 frames each 128x128 pixels and 1200 energy channels.

In [ ]:
dc = JEOL_pts(base / "data" / "128.pts", read_drift="yes", split_frames=True, E_cutoff=12.0)
dc.dcube.shape

Before we continue, check, if drift corrections are required. We have two options: Make a mp4 movie with the drift image in the left pane and the total edx intensity in the right pane. Use the file name of the data but with `.pts` replaced by `mp4`. Check out the movie to realize that the sample jumps around a bit. In the output below, only the last frame is shown.

In [ ]:
dc.make_movie()

Alternatively, calculate the image shift for each frame (from cross correlation with a reference) using the drift images (`align_src="drift_images"`) or data (`align_src="data"`). We might apply a Wiener filter (`filtered=True`) before performing the cross correlation. We use the 10st frame (`reference=9`) as reference. Ignore any warnings.

In [ ]:
dc.shifts(align_src="data", filtered=False, reference=9)

We could also visualize the shifts as 2D probability distribution (histogram).

In [ ]:
m, e = dc.drift_statistics(align_src="data", filtered=False, reference=9)
plt.imshow(m, extent=e)

Apply the drift correction. **Careful**, this overwrites the data cube (`JEOL_pts.dcube`) and the drift images (`JEOL_pts.drift_images`). Make a movie again (this time, specify a file name) to observe the result.

In [ ]:
dc.correct_drift(align_src="data", apply_filter=False, reference=9)
dc.make_movie(base / "corrected.mp4")

Check if carbon writing begins at some point during data acquisition. Follow the intensity of the carbon Ka line between 0.225 keV and 0.325 keV (manually determined from a zoomed-in version of the reference spectrum).

In [ ]:
I_carbon = dc.time_series(interval=(0.225, 0.325), energy=True)
JU.plot_tseries(I_carbon, M_ticks=(9, None))

Extract an elemental map for Ni using the Ka lines at 7.5 keV.

In [ ]:
Ni = dc.map(interval=(7.36, 7.6), energy=True)
JU.plot_map(Ni, "purple", label="Ni")

Try to improve the map by adding the contributions of the Kb and the L lines. Note, that this also adds substantial amounts of noise.

In [ ]:
Ni += dc.map(interval=(8.2, 8.4), energy=True)
Ni += dc.map(interval=(0.8, 0.9), energy=True)
JU.plot_map(Ni, "purple", label="Ni")

Overlay the map onto the reference drift image to help with the interpretation. Use brighter color.

In [ ]:
JU.create_overlay([Ni], ["yellow"], BG_image=dc.drift_images[9])

Extract the spectra of two regions, one containing a single Ni nanoparticle (`circle`) and one containing the empty support (`square`). Show the regions for verification and then plot the corresponding spectrum. Only use frames 0 to 39.

In [ ]:
circle = (96, 43, 10)
circle_spec = dc.spectrum(ROI=circle, frames=list(range(40)))
JU.show_ROI(dc.drift_images[9], circle, cmap="Greys")

In [ ]:
square = (43, 58, 26, 40)
square_spec = dc.spectrum(ROI=square, frames=list(range(40)))
JU.show_ROI(dc.drift_images[9], square, cmap="Greys")

In [ ]:
JU.plot_spectrum([circle_spec, square_spec], legends=["circle", "square"])

Finally, extract some profiles along an arbitrary line (on an image or map). First, verify line definition then extract Ni and and O profiles and plot them. For this we reuse the nickel elemental map and create an oxygen map using Ka at 0.5 keV.

In [ ]:
line = (20, 34, 120, 22)
JU.show_line(dc.drift_images[9], line, linewidth=5)

In [ ]:
x, pNi = JU.get_profile(Ni, line, linewidth=10, reduce_func=np.sum)

O = dc.map(interval=(0.48, 0.575), energy=True)
_, pO = JU.get_profile(O, line, linewidth=10, reduce_func=np.sum)

JU.plot_profile([x, x], [pO, pNi], legends=["O", "Ni"])